The LLM is allowed to propose a decision, but it is not allowed to directly execute the action. The workflow pauses for human approval, and only after approval does the application execute the action

```
User Request
      ↓
     LLM
      ↓
Structured Proposed Action

{
  action: "refund",
  customer: "CUST-101",
  amount: 5000,
  reason: "Customer received a damaged product"
}

      ↓
 HUMAN REVIEW
      ↓

Approve? yes / no

     /      \
   yes       no
    ↓         ↓
 Execute     Stop
 Refund

 ```

In [1]:

from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI


In [2]:
model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


In [3]:
class ProposedAction(BaseModel):
    action: Literal[
        "refund",
        "reject",
        "manual_review"
    ]

    customer: str = Field(
        description="Customer ID"
    )

    amount: float = Field(
        description="Refund amount"
    )

    reason: str = Field(
        description="Reason for the proposed action"
    )

In [4]:
structured_model = model.with_structured_output(ProposedAction)

In [5]:
def ai_assistant(user_request):

    prompt = f"""
You are a customer support AI agent.

Analyze the following customer request:

{user_request}

Decide the appropriate action.

Possible actions:
- refund
- reject
- manual_review

Return:
- action
- customer ID
- refund amount
- reason

Do not execute the action.
Only propose the action for human approval.
"""

    proposed_action = structured_model.invoke(
        prompt
    )

    return proposed_action

In [6]:
def human_review(action):

    print("\n========== AI Proposed Action ==========")

    print("Action   :", action.action)
    print("Customer :", action.customer)
    print("Amount   :", action.amount)
    print("Reason   :", action.reason)

    print("========================================")


    decision = input(
        "\nApprove this action? (yes/no): "
    ).strip().lower()


    if decision == "yes":
        return True

    return False

In [7]:
def execute_action(action):

    if action.action == "refund":

        print(
            f"\nRefund of ₹{action.amount} "
            f"processed for {action.customer}."
        )


    elif action.action == "reject":

        print(
            f"\nRequest rejected for "
            f"{action.customer}."
        )


    elif action.action == "manual_review":

        print(
            f"\nRequest for {action.customer} "
            f"sent for manual review."
        )


In [8]:
def run():

    user_request = input(
        "User Request: "
    )


    # Step 1
    # LLM analyzes request and proposes action

    proposed_action = ai_assistant(
        user_request
    )


    # Step 2
    # Human reviews the AI decision

    approved = human_review(
        proposed_action
    )


    # Step 3
    # Execute only after human approval

    if approved:

        execute_action(
            proposed_action
        )

    else:

        print(
            "\nAction rejected by human."
        )

In [12]:
run()


========== AI Proposed Action ==========
Action   : reject
Customer : CUST-202
Amount   : 5000.0
Reason   : The purchase was made 8 months ago, which is outside the refund policy period.

Action rejected by human.
